# Hafta 2 - Adım 1: Veri çekme (SMF, sistem yönü, GİP + Kasım güncellemesi)

**Ne çeker?**
1. PTF, gerçek zamanlı üretim ve tüketim: Hafta 1 önbelleği (`../Hafta_01/01_ham_veri`) kullanılır, sadece eksik aylar (Kasım 2026) indirilir.
2. Yeni veri setleri: **SMF**, **sistem yönü**, **GİP ağırlıklı ortalama fiyat** (Oca 2023 - dün).

**Klasör yapısı:** Bu dosyayı `Elektrik_Piyasasi_24_Haftalik_Analiz/Hafta_02/` içine koyun. Hafta_01 klasörü yanında durmalı.

**Güvenlik:** Kullanıcı adı ve şifre koda yazılmaz, çalıştırınca sorulur. Paylaşmadan önce Jupyter'de *Clear All Outputs* yapın.

**Çalıştırma:** Baştan sona çalıştırın (Kernel > Restart & Run All). SMF, sistem yönü ve GİP ilk seferde 2023'ten itibaren iner, birkaç dakika sürebilir. Sonraki çalıştırmalarda sadece eksik aylar iner.

In [ ]:
# 1) Kurulum (bir kez)
# !pip install requests pandas pyarrow matplotlib statsmodels

In [ ]:
# 2) Ayarlar
import os, time, json, getpass
from datetime import date, timedelta
from pathlib import Path
import requests, pandas as pd

DENEME_MODU = False   # True: sadece Ekim-Kasım 2026 (hızlı bağlantı testi)

BUGUN = date.today()
DUN   = BUGUN - timedelta(days=1)
BASLANGIC = date(2026, 10, 1) if DENEME_MODU else date(2023, 1, 1)
BITIS = DUN

H01_HAM = Path("../Hafta_01/01_ham_veri")          # Hafta 1 önbelleği (varsa yeniden indirilmez)
HAM  = Path("01_ham_veri");  HAM.mkdir(exist_ok=True)
ISLN = Path("02_islenmis");  ISLN.mkdir(exist_ok=True)

LOGIN_URL = "https://giris.epias.com.tr/cas/v1/tickets"
API_ROOT  = "https://seffaflik.epias.com.tr/electricity-service"

# Her veri seti için bir veya birden fazla aday uç nokta (ilk çalışan kullanılır)
ENDPOINTS = {
    "ptf":         ["/v1/markets/dam/data/mcp"],
    "uretim":      ["/v1/generation/data/realtime-generation"],
    "tuketim":     ["/v1/consumption/data/realtime-consumption"],
    "smf":         ["/v1/markets/bpm/data/system-marginal-price"],
    "sistem_yonu": ["/v1/markets/bpm/data/system-direction"],
    "gip":         ["/v1/markets/idm/data/weighted-average-price", "/v1/markets/idm/data/wap"],
}
ZORUNLU = ["ptf", "uretim", "tuketim", "smf", "sistem_yonu"]
OPSIYONEL = ["gip"]   # uç nokta bulunamazsa uyarı verip devam eder
print("Aralık:", BASLANGIC, "-", BITIS)

In [ ]:
# 3) Giriş: TGT (Ticket Granting Ticket) alma
# Kullanıcı adınız kayıt e-postanızdır. Bilgiler ekrana yazılmaz, kaydedilmez.
class Oturum:
    def __init__(self):
        self.kullanici = os.environ.get("EPIAS_USER") or input("EPİAŞ kullanıcı adı (e-posta): ").strip()
        self._sifre    = os.environ.get("EPIAS_PASS") or getpass.getpass("EPİAŞ şifre: ")
        self.tgt = None
        self.tgt_zamani = 0

    def yenile(self):
        r = requests.post(
            LOGIN_URL,
            data={"username": self.kullanici, "password": self._sifre},
            headers={"Content-Type": "application/x-www-form-urlencoded", "Accept": "text/plain"},
            timeout=30,
        )
        if r.status_code not in (200, 201) or not r.text.strip().startswith("TGT-"):
            raise RuntimeError(f"Giriş başarısız (HTTP {r.status_code}). Kullanıcı adı/şifreyi kontrol edin.")
        self.tgt = r.text.strip()
        self.tgt_zamani = time.time()

    def baslik(self):
        # Bilet birkaç saat geçerlidir; 2 saatte bir yenilemek güvenli tarafta kalır
        if self.tgt is None or time.time() - self.tgt_zamani > 2 * 3600:
            self.yenile()
        return {"TGT": self.tgt, "Content-Type": "application/json", "Accept": "application/json"}

oturum = Oturum()
oturum.yenile()
print("Giriş başarılı.")

In [ ]:
# 4) Aylık parçalarla çekme. Tamamlanmamış ay (bu ay) önbelleğe alınmaz, her seferinde yenilenir.
def ay_araliklari(bas, bit):
    d = date(bas.year, bas.month, 1)
    while d <= bit:
        sonraki = date(d.year + (d.month // 12), (d.month % 12) + 1, 1)
        yield max(d, bas), min(sonraki - timedelta(days=1), bit), (sonraki - timedelta(days=1)) <= bit
        d = sonraki

def iso(d): return f"{d.isoformat()}T00:00:00+03:00"

class UcNoktaYok(Exception): pass

def api_cagri(yol, bas, bit, deneme=4):
    govde = {"startDate": iso(bas), "endDate": iso(bit)}
    for i in range(deneme):
        try:
            r = requests.post(API_ROOT + yol, headers=oturum.baslik(), json=govde, timeout=90)
            if r.status_code == 404: raise UcNoktaYok(yol)
            if r.status_code in (401, 403): oturum.yenile(); continue
            if r.status_code == 429 or r.status_code >= 500: time.sleep(2 ** (i + 1)); continue
            r.raise_for_status(); return r.json()
        except requests.exceptions.RequestException:
            if i == deneme - 1: raise
            time.sleep(2 ** (i + 1))
    raise RuntimeError(f"Çağrı başarısız: {yol} {bas}-{bit}")

def onbellek(ad, bas):
    for k in (H01_HAM / f"{ad}_{bas:%Y-%m}.json", HAM / f"{ad}_{bas:%Y-%m}.json"):
        if k.exists(): return json.loads(k.read_text(encoding="utf-8"))
    return None

def veri_seti_cek(ad):
    parcalar, yol_idx = [], 0
    for bas, bit, tam_ay in ay_araliklari(BASLANGIC, BITIS):
        veri = onbellek(ad, bas) if tam_ay else None
        while veri is None:
            try:
                veri = api_cagri(ENDPOINTS[ad][yol_idx], bas, bit)
            except UcNoktaYok:
                yol_idx += 1
                if yol_idx >= len(ENDPOINTS[ad]): raise
                continue
            dosya = HAM / (f"{ad}_{bas:%Y-%m}.json" if tam_ay else f"{ad}_{bas:%Y-%m}_kismi.json")
            dosya.write_text(json.dumps(veri, ensure_ascii=False), encoding="utf-8"); time.sleep(0.5)
        k = veri.get("items", [])
        if k:
            df = pd.DataFrame(k); df["_ay"] = f"{bas:%Y-%m}"; parcalar.append(df)
        print(f"{ad} {bas:%Y-%m}{'' if tam_ay else ' (kısmi)'}: {len(k)} kayıt")
    return pd.concat(parcalar, ignore_index=True) if parcalar else pd.DataFrame()

ham = {}
for ad in ZORUNLU + OPSIYONEL:
    try:
        ham[ad] = veri_seti_cek(ad)
    except UcNoktaYok:
        if ad in OPSIYONEL:
            print(f"UYARI: {ad} için uç nokta bulunamadı, bu veri atlanıyor."); ham[ad] = pd.DataFrame()
        else: raise

In [ ]:
# 5) Sütun adlarını kontrol edin (yeni veri setlerinde ilk kez görüyoruz)
for ad, df in ham.items():
    print("="*60); print(ad, df.shape); print(list(df.columns))
    if len(df): display(df.head(3))

In [ ]:
# 6) Saatlik tabloya çevirme
def zaman_damgasi(df):
    d = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Europe/Istanbul")
    for kol in ("hour", "time"):
        if kol in df.columns and d.dt.hour.nunique() == 1:
            saat = pd.to_numeric(df[kol].astype(str).str.slice(0, 2), errors="coerce").fillna(0)
            d = d + pd.to_timedelta(saat, unit="h"); break
    return d.dt.tz_localize(None)

def hazirla(df, kolonlar):
    if df.empty: return pd.DataFrame()
    out = df.copy(); out["zaman"] = zaman_damgasi(out)
    out = out.drop_duplicates("zaman").set_index("zaman").sort_index()
    return out[[k for k in kolonlar if k in out.columns]]

def sayisal_kolon(df, tercihler):
    """Tercih edilen adlardan ilk bulunanı, yoksa ilk sayısal sütunu döndürür."""
    for t in tercihler:
        if t in df.columns: return t
    adaylar = [c for c in df.columns if c not in ("date", "hour", "time", "_ay") and pd.api.types.is_numeric_dtype(df[c])]
    return adaylar[0] if adaylar else None

ptf = hazirla(ham["ptf"], ["price", "priceUsd", "priceEur"]).rename(columns={"price": "ptf", "priceUsd": "ptf_usd", "priceEur": "ptf_eur"})
tuk = hazirla(ham["tuketim"], ["consumption"]).rename(columns={"consumption": "tuketim_mwh"})
UK = ["total","naturalGas","dammedHydro","lignite","river","importCoal","wind","sun","fueloil","geothermal",
      "asphaltiteCoal","blackCoal","biomass","naphta","lng","importExport","wasteheat"]
uret = hazirla(ham["uretim"], UK).add_prefix("uretim_").rename(columns={"uretim_sun": "uretim_solar"})
ana = ptf.join([tuk, uret], how="outer")

# SMF
k = sayisal_kolon(ham["smf"], ["systemMarginalPrice", "smp", "price"])
print("SMF sütunu:", k)
ana = ana.join(hazirla(ham["smf"], [k]).rename(columns={k: "smf"}))

# Sistem yönü: metin sütununu açık / fazla / denge olarak sınıflandır
sy = ham["sistem_yonu"]
TR = str.maketrans("ÇĞİIÖŞÜçğıiöşü", "CGIIOSUcgiiosu")
def yon_sinifi(x):
    s = str(x).translate(TR).upper()
    if "ACIK" in s or "ACIG" in s or "DEFICIT" in s or s in ("YAL", "UP"): return "acik"
    if "FAZLA" in s or "SURPLUS" in s or s in ("YAT", "DOWN"): return "fazla"
    if "DENGE" in s or "BALANCE" in s: return "denge"
    return None
metin = [c for c in sy.columns if c not in ("date", "hour", "time", "_ay") and not pd.api.types.is_numeric_dtype(sy[c])
         and sy[c].map(yon_sinifi).notna().mean() > 0.5]
print("Sistem yönü metin sütunu:", metin[:1], "| değerler:", sy[metin[0]].unique()[:6] if metin else "-")
# Sayısal miktar sütunu (varsa). "...Id" gibi kod sütunları miktar değildir, alınmaz.
miktar = [c for c in sy.columns if pd.api.types.is_numeric_dtype(sy[c]) and not c.lower().endswith("id")]
sy_df = hazirla(sy, metin[:1] + miktar[:1])
if metin:
    sy_df["sistem_yonu"] = sy_df[metin[0]].map(yon_sinifi); sy_df = sy_df.drop(columns=metin[0])
sy_df = sy_df.rename(columns={c: "sy_miktar" for c in sy_df.columns if c != "sistem_yonu"})
ana = ana.join(sy_df)

# GİP ağırlıklı ortalama fiyat (varsa)
if len(ham.get("gip", [])):
    k = sayisal_kolon(ham["gip"], ["wap", "weightedAveragePrice", "price"]); print("GİP sütunu:", k)
    ana = ana.join(hazirla(ham["gip"], [k]).rename(columns={k: "gip"}))

ana.index.name = "zaman"
print(ana.shape); display(ana.tail(3))

In [ ]:
# 7) Kalite kontrol
beklenen = pd.date_range(f"{BASLANGIC} 00:00", f"{BITIS} 23:00", freq="h")
print("Beklenen saat:", len(beklenen), "| Tabloda:", len(ana), "| Eksik:", len(beklenen.difference(ana.index)))
print("\nAylık doluluk (%), yeni sütunlar:")
yeni = [c for c in ["smf", "sistem_yonu", "sy_miktar", "gip"] if c in ana.columns]
display((ana[yeni].notna().groupby(ana.index.to_period("M")).mean() * 100).round(0).tail(14))
if "sistem_yonu" in ana:
    print("Sistem yönü dağılımı (%):"); print((ana.sistem_yonu.value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
# 8) Kaydet
ana.to_csv(ISLN / "ana_tablo_h02_saatlik.csv", encoding="utf-8-sig")
print("Kaydedildi:", ISLN / "ana_tablo_h02_saatlik.csv")
if DENEME_MODU:
    print("\n" + "!" * 70 + "\nDENEME_MODU = True: tabloda sadece Ekim-Kasım 2026 var. Analiz için YETERSİZ.\n"
          "2. hücrede DENEME_MODU = False yapıp bu not defterini baştan tekrar çalıştırın.\n" + "!" * 70)
else:
    print("Tam veri hazır. Şimdi 02_hafta02_analiz.ipynb'yi çalıştırın.")